# Task 1 · Cache Reuse Rule
**Owner:** Naveena Sivapuram · DSA 4

Which matching rule saves the most LLM calls without reusing answers for look-alike questions?

Kaggle settings: **GPU T4** (only for the embeddings), **Internet on**. Add Input → **dsa4-data** and **dsa4-cache**.

## 1. Introduction

*Write this in your own words from the Task 1 page of the Modeling Tasks document: the problem, the research question, the hypothesis and the four papers.*

## 2. Setup and data

In [ ]:
# Setup: clone the team repository and install what Kaggle lacks
import os, subprocess, sys
REPO_URL = "https://github.com/Naveena126/dsa4-reuse-retrieve-generate.git"
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "bitsandbytes>=0.43"], check=True)
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "src")
import glob
def find_input(name):
    # Kaggle mounts datasets at /kaggle/input/<name> or /kaggle/input/datasets/<owner>/<name>
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(h)]
    if not hits:
        raise FileNotFoundError(f"Add the dataset {name} with Add Input")
    return hits[0]
print("ready")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from dsa4 import data, stream, reuse

DATA = find_input("dsa4-data")
CACHE = glob.glob(find_input("dsa4-cache") + "/**/cache.jsonl", recursive=True)[0]
MET, FIG = Path("results/metrics"), Path("results/figures")
MET.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

q = data.load_questions(DATA)
cache = pd.read_json(CACHE, lines=True)
cache = cache[cache["variants"].notna()]          # questions whose variants were generated
var = stream.variants_table(cache)
q = q[q["qid"].isin(cache["qid"])].reset_index(drop=True)
print(len(q), "source questions;", var["kind"].value_counts().to_dict())

In [ ]:
# Drop look-alikes that Qwen returned unchanged (identical to the source or to a paraphrase)
from dsa4.text import normalize
src_norm = dict(zip(q["qid"], q["question"].map(normalize)))
para_norm = var[var["kind"] == "paraphrase"].groupby("qid")["text"].apply(lambda s: set(map(normalize, s)))
is_copy = var.apply(lambda r: r["kind"] == "lookalike" and
                    (normalize(r["text"]) == src_norm.get(r["qid"]) or
                     normalize(r["text"]) in para_norm.get(r["qid"], set())), axis=1)
print(int(is_copy.sum()), "look-alikes identical to their source or a paraphrase; dropped")
var = var[~is_copy].reset_index(drop=True)

## 3. Data exploration

In [ ]:
acc = pd.DataFrame({"dataset": cache["dataset"], "direct": cache["direct"].str["correct"],
                    "rag": cache["rag"].str["correct"]})
display(acc.groupby("dataset")[["direct", "rag"]].mean().round(3).rename(columns=lambda c: c + " accuracy"))
display(var[var["kind"] == "lookalike"]["changed"].value_counts().rename("look-alikes by type of change"))
pd.set_option("display.max_colwidth", 90)
ex = var[var["qid"] == var["qid"].iloc[0]]
print("Source:", q.set_index("qid").loc[ex["qid"].iloc[0], "question"])
display(ex[["kind", "text"]])

**Hand-check.** *Add the agreement rate from the 100 hand-checked variants here once the shared sheet is complete (share of paraphrases that really keep the meaning, and of look-alikes that really change the answer).*

## 4. Embeddings

Every source question and variant is embedded once with each model (normalised vectors, so the dot product is the cosine similarity).

In [ ]:
from sentence_transformers import SentenceTransformer

texts = pd.concat([q[["qid", "question"]].rename(columns={"question": "text"}).assign(vid=q["qid"]),
                   var[["vid", "text"]]], ignore_index=True).drop_duplicates("vid")
MODELS = {"minilm": "sentence-transformers/all-MiniLM-L6-v2", "bge": "BAAI/bge-base-en-v1.5"}
EMB = {}
for name, mid in MODELS.items():
    m = SentenceTransformer(mid)
    v = m.encode(texts["text"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=True)
    EMB[name] = dict(zip(texts["vid"], v.astype("float32")))
print({k: len(v) for k, v in EMB.items()})

## 5. Method

The stream replays each split's source questions, then their paraphrases and look-alikes at random later positions. A query reuses the closest cached entry if the rule says so; otherwise it is answered and stored. A reuse is **safe** only if both questions have the same correct answer.

- **calls saved** = reuses ÷ all queries
- **unsafe reuse** = unsafe reuses ÷ all reuses (our 2% target)
- **safe recall** = safe reuses ÷ queries that had a safe entry available

Thresholds are chosen on the validation split (most calls saved with unsafe reuse ≤ 2%) and reported on the test split, over seeds 0, 1 and 2.

In [ ]:
THRESH = np.round(np.arange(0.60, 1.0001, 0.01), 2)
TARGET = 0.02
RULES = {"V1 MiniLM": "minilm", "V1 bge": "bge", "V2 ensemble": "ensemble"}

def make_stream(seed, split, datasets=("triviaqa", "hotpotqa"), repeat=0.0):
    src = q[(q[f"split_s{seed}"] == split) & q["dataset"].isin(datasets)]
    return stream.build_stream(src[["qid", "question"]], var[var["qid"].isin(src["qid"])],
                               repeat_rate=repeat, seed=seed)

def sim_for(s, rule):
    if rule == "ensemble":
        return (sim_for(s, "minilm") + sim_for(s, "bge")) / 2
    return reuse.similarity_matrix(np.stack([EMB[rule][v] for v in s["vid"]]))

s = make_stream(0, "val")
print(len(s), "events in the seed-0 validation stream:", s["kind"].value_counts().to_dict())

## 6. Results: exact match, V1 and V2

In [ ]:
rows, curves = [], {}
for seed in (0, 1, 2):
    sv, st = make_stream(seed, "val"), make_stream(seed, "test")
    rows.append({"rule": "V0 exact match", "seed": seed, "threshold": np.nan,
                 **reuse.summarise(reuse.simulate_exact(st))})
    for label, rule in RULES.items():
        cv = reuse.sweep(sv, sim_for(sv, rule), THRESH)
        curves[(label, seed)] = cv
        t = reuse.pick_threshold(cv, TARGET)
        rows.append({"rule": label, "seed": seed, "threshold": t,
                     **reuse.summarise(reuse.simulate(st, sim_for(st, rule), t))})
res = pd.DataFrame(rows)
res.to_csv(MET / "task1_results_by_seed.csv", index=False)
summary = res.groupby("rule")[["threshold", "calls_saved", "unsafe_reuse", "safe_recall"]].agg(["mean", "std"]).round(3)
summary.to_csv(MET / "task1_summary.csv")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for label in RULES:
    cv = curves[(label, 0)]
    ax.plot(cv["unsafe_reuse"] * 100, cv["calls_saved"] * 100, marker=".", label=label)
ax.axvline(TARGET * 100, color="grey", ls="--", lw=1)
ax.set_xlabel("Unsafe reuse (% of reuses)"); ax.set_ylabel("Calls saved (% of queries)")
ax.set_title("Savings vs. risk on validation (seed 0)"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "task1_savings_vs_risk.png", dpi=150)

In [ ]:
# Why thresholds fail: similarity of each variant to its own source question
rows = []
for name in ("minilm", "bge"):
    for r in var.itertuples():
        rows.append({"model": name, "kind": r.kind, "sim": float(EMB[name][r.vid] @ EMB[name][r.qid])})
sims = pd.DataFrame(rows)
display(sims.groupby(["model", "kind"])["sim"].describe()[["mean", "25%", "50%", "75%"]].round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, name in zip(axes, ("minilm", "bge")):
    for kind in ("paraphrase", "lookalike"):
        ax.hist(sims[(sims["model"] == name) & (sims["kind"] == kind)]["sim"], bins=30, alpha=.6, label=kind)
    ax.set_title(name); ax.set_xlabel("Cosine similarity to the source question")
axes[0].set_ylabel("Variants"); axes[0].legend()
fig.tight_layout(); fig.savefig(FIG / "task1_similarity_by_kind.png", dpi=150)

In [ ]:
# Same rules on a stream where 20% of queries are exact repeats
rows20 = []
for seed in (0, 1, 2):
    st = make_stream(seed, "test", repeat=0.2)
    for label, rule in RULES.items():
        t = res[(res["rule"] == label) & (res["seed"] == seed)]["threshold"].iloc[0]
        rows20.append({"rule": label, "seed": seed, **reuse.summarise(reuse.simulate(st, sim_for(st, rule), t))})
pd.DataFrame(rows20).groupby("rule")[["calls_saved", "unsafe_reuse"]].agg(["mean", "std"]).round(3)

## 7. Transfer: TriviaQA → HotpotQA

Threshold tuned on TriviaQA validation, applied without retuning to HotpotQA test, compared with tuning on HotpotQA itself.

In [ ]:
rows_t = []
for seed in (0, 1, 2):
    tq_val = make_stream(seed, "val", ("triviaqa",))
    hq_val, hq_test = make_stream(seed, "val", ("hotpotqa",)), make_stream(seed, "test", ("hotpotqa",))
    for label, rule in RULES.items():
        t_tq = reuse.pick_threshold(reuse.sweep(tq_val, sim_for(tq_val, rule), THRESH), TARGET)
        t_hq = reuse.pick_threshold(reuse.sweep(hq_val, sim_for(hq_val, rule), THRESH), TARGET)
        for tuned, t in (("TriviaQA (transfer)", t_tq), ("HotpotQA (in-domain)", t_hq)):
            rows_t.append({"rule": label, "tuned_on": tuned, "seed": seed, "threshold": t,
                           **reuse.summarise(reuse.simulate(hq_test, sim_for(hq_test, rule), t))})
transfer = pd.DataFrame(rows_t)
transfer.to_csv(MET / "task1_transfer.csv", index=False)
transfer.groupby(["rule", "tuned_on"])[["threshold", "calls_saved", "unsafe_reuse"]].mean().round(3)

## 8. Error analysis

Which unsafe reuses does each rule make? Look-alikes are grouped by what was changed; "different question" means a source or paraphrase matched an unrelated cached question.

In [ ]:
vmeta = var.set_index("vid")
def failures(seed, label):
    st = make_stream(seed, "test")
    t = res[(res["rule"] == label) & (res["seed"] == seed)]["threshold"].iloc[0]
    r = reuse.simulate(st, sim_for(st, RULES[label]), t)
    bad = st[(r["reused"]) & (r["safe"] == False)].copy()
    bad["matched_text"] = st["text"].to_numpy()[r.loc[bad.index, "match"].to_numpy()]
    bad["similarity"] = r.loc[bad.index, "best_sim"].round(3)
    bad["type"] = [vmeta.loc[v, "changed"] if k == "lookalike" else "different question"
                   for k, v in zip(bad["kind"], bad["vid"])]
    return bad

err = pd.concat([failures(0, label).assign(rule=label) for label in RULES], ignore_index=True)
err.to_csv(MET / "task1_unsafe_reuses_seed0.csv", index=False)
display(pd.crosstab(err["rule"], err["type"]))
err[["rule", "type", "similarity", "text", "matched_text"]].head(10)